# Stochastic Calculus, Step by Step · Lesson 33: Complex numbers and Fourier transforms

Companion notebook for the lesson. Every number in the lesson's text, tables and figures is produced by running this notebook and read from `build/outputs.json`.

> **The course's numbers.** A share priced at S₀ = 100, a strike K = 105, a risk-free rate r = 3% (continuously compounded), a volatility σ = 20% and a horizon T = 1 year. Where a lesson needs the share's real-world expected return, it is μ = 8%. Made up for teaching; every random number is simulated with a fixed seed.
>
> Educational material only, not financial advice.

1. Setup · 2. Complex numbers and Euler's formula · 3. The characteristic function · 4. From characteristic function to density · 5. Pricing the call by Fourier inversion · 6. The complex-log trap · 7. Exercises · 8. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S0, K, r, sigma, T = 100.0, 105.0, 0.03, 0.20, 1.0     # the course's numbers
N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))   # standard normal distribution function
out = {"meta": {
    "series": "Stochastic Calculus, Step by Step",
    "lesson": 33,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}, "inputs": {"S0": S0, "K": K, "r": r, "sigma": sigma, "T": T, "r_pct": r * 100, "sigma_pct": sigma * 100}}

In [ ]:
mu = 0.08                         # the share's real-world expected return
theta_mpr = (mu - r) / sigma      # the market price of risk
Nv = np.vectorize(N)              # N, for arrays
phi = lambda x: np.exp(-np.asarray(x, float) ** 2 / 2) / math.sqrt(2 * math.pi)   # the normal density


def d_plus(s, tau, sig=sigma, k=K):
    return (np.log(s / k) + (r + 0.5 * sig**2) * tau) / (sig * np.sqrt(tau))


def d_minus(s, tau, sig=sigma, k=K):
    return d_plus(s, tau, sig, k) - sig * np.sqrt(tau)


def call(s, tau, sig=sigma, k=K):
    """The Black-Scholes call price: Lesson 20 derives it."""
    return s * Nv(d_plus(s, tau, sig, k)) - k * np.exp(-r * tau) * Nv(d_minus(s, tau, sig, k))


def put(s, tau, sig=sigma, k=K):
    return call(s, tau, sig, k) - s + k * np.exp(-r * tau)


out["bs"] = {"mu": mu, "theta": theta_mpr, "d_plus": float(d_plus(S0, T)), "d_minus": float(d_minus(S0, T)),
             "N_plus": N(float(d_plus(S0, T))), "N_minus": N(float(d_minus(S0, T))),
             "call": float(call(S0, T)), "put": float(put(S0, T))}
out["bs"]

## 2. Complex numbers and Euler's formula

A complex number $z = a + bi$, with $i^2 = -1$, is a point in the plane. Multiplying by $i$ turns it a quarter turn. Euler's formula $e^{i\theta} = \cos\theta + i\sin\theta$ follows from the Taylor series of $e^x$ (Lesson 1). Python's `complex` type does the arithmetic. The partial sums of $\sum(i\pi)^n/n!$ spiral in to $e^{i\pi} = -1$.

In [ ]:
z = 3 + 4j
w = 1 - 2j
partial, term, sums = 0j, 1 + 0j, []
for n in range(22):
    partial += term
    sums.append(partial)
    term *= 1j * math.pi / (n + 1)
out["complex"] = {"prod_re": (z * w).real, "prod_im": (z * w).imag, "mod": abs(z), "arg_deg": math.degrees(math.atan2(z.imag, z.real)),
                  "inv_re": (1 / z).real, "inv_im": (1 / z).imag, "iz_re": (1j * z).real, "iz_im": (1j * z).imag,
                  "euler_pi_re": complex(np.exp(1j * math.pi)).real, "sum8_re": sums[8].real, "sum8_im": sums[8].imag,
                  "sum20_err": abs(sums[20] + 1),
                  "fig": {"re": [c.real for c in sums[:16]], "im": [c.imag for c in sums[:16]]}}
{k: v for k, v in out["complex"].items() if k != "fig"}

## 3. The characteristic function

The **characteristic function** of $X$ is $\varphi(u) = E[e^{iuX}]$. It always exists, since $|e^{iuX}| = 1$. For a standard normal it is $e^{-u^2/2}$: a simulation agrees. For the log price, $X = \ln S_T$ under the risk-neutral probability, it is

$$\varphi(u) = \exp\big(iu\,m - \tfrac12v\,u^2\big), \qquad m = \ln S_0 + (r - \tfrac12\sigma^2)T, \quad v = \sigma^2T .$$

In [ ]:
rng = np.random.default_rng(seed)
Zs = rng.standard_normal(1_000_000)
cf_rows = [{"u": u, "sim_re": float(np.cos(u * Zs).mean()), "sim_im": float(np.sin(u * Zs).mean()), "exact": math.exp(-u**2 / 2)}
           for u in (0.5, 1.0, 2.0)]
m_x = math.log(S0) + (r - 0.5 * sigma**2) * T
v_x = sigma**2 * T
cf = lambda u: np.exp(1j * u * m_x - 0.5 * v_x * u**2)
uu = np.linspace(0, 15, 301)
out["cf"] = {"rows": cf_rows, "m": m_x, "v": v_x, "lnS0": math.log(S0), "lnK": math.log(K),
             "fig": {"u": uu.tolist(), "re": cf(uu).real.round(5).tolist(), "im": cf(uu).imag.round(5).tolist(), "mod": np.abs(cf(uu)).round(5).tolist()}}
pd.DataFrame(cf_rows).round(4)

## 4. From characteristic function to density

**Fourier inversion** recovers the density: $f(x) = \frac1\pi\int_0^\infty \mathrm{Re}\big[e^{-iux}\varphi(u)\big]\,du$. We do the integral numerically (a midpoint sum) and compare with the normal density of $\ln S_T$.

In [ ]:
du, U = 0.01, 60.0
ug = np.arange(du / 2, U, du)
cf_g = cf(ug)
xs = np.linspace(m_x - 4 * sigma, m_x + 4 * sigma, 161)
dens = np.array([(np.exp(-1j * ug * x) * cf_g).real.sum() * du / math.pi for x in xs])
exact_d = np.exp(-(xs - m_x) ** 2 / (2 * v_x)) / math.sqrt(2 * math.pi * v_x)
out["dens"] = {"du": du, "U": U, "max_err": float(np.abs(dens - exact_d).max()), "peak": float(exact_d.max()),
               "fig": {"x": xs.round(4).tolist(), "inv": dens.round(5).tolist(), "exact": exact_d.round(5).tolist()}}
out["dens"]["max_err"]

## 5. Pricing the call by Fourier inversion

The **Gil-Pelaez** formula turns a characteristic function into a probability:

$$P(X > k) = \frac12 + \frac1\pi\int_0^\infty \mathrm{Re}\Big[\frac{e^{-iuk}\varphi(u)}{iu}\Big]du .$$

With $k = \ln K$ it gives $\tilde P(S_T > K) = N(d_-)$. Under the share measure of Lesson 27, the characteristic function becomes $\varphi(u - i)/\varphi(-i)$, and the same formula gives $N(d_+)$. Then $C = S_0N(d_+) - Ke^{-rT}N(d_-)$, with no density in sight. We also measure how the answer depends on where the integral stops ($U$) and on the step ($du$).

In [ ]:
def gil_pelaez(phi_, k, U=60.0, du=0.01):
    u = np.arange(du / 2, U, du)
    return 0.5 + (np.exp(-1j * u * k) * phi_(u) / (1j * u)).real.sum() * du / math.pi


cf_share = lambda u: cf(u - 1j) / cf(-1j)
k_ = math.log(K)
P2 = gil_pelaez(cf, k_)
P1 = gil_pelaez(cf_share, k_)
call_f = S0 * P1 - K * math.exp(-r * T) * P2
trunc_rows = [{"U": U_, "call": S0 * gil_pelaez(cf_share, k_, U_) - K * math.exp(-r * T) * gil_pelaez(cf, k_, U_)} for U_ in (2.0, 5.0, 10.0, 20.0)]
step_rows = [{"du": d_, "points": len(np.arange(d_ / 2, 60, d_)), "alias": 2 * math.pi / d_,
              "call": S0 * gil_pelaez(cf_share, k_, 60, d_) - K * math.exp(-r * T) * gil_pelaez(cf, k_, 60, d_)} for d_ in (10.0, 8.0, 6.0, 4.0, 2.0)]
for row in trunc_rows + step_rows:
    row["error"] = row["call"] - float(call(S0, T))
out["gp"] = {"P2": P2, "P1": P1, "N_minus": out["bs"]["N_minus"], "N_plus": out["bs"]["N_plus"], "call": call_f, "bs_call": float(call(S0, T)),
             "error": call_f - float(call(S0, T)), "cf_minus_i": float(cf(-1j).real), "forward": S0 * math.exp(r * T),
             "limit0": m_x - k_, "trunc": trunc_rows, "step": step_rows}
pd.DataFrame(trunc_rows + step_rows)

## 6. The complex-log trap

A complex number has many logarithms: $\ln z = \ln|z| + i(\arg z + 2\pi n)$. Computers return the one with $\arg z$ in $(-\pi, \pi]$. A power $z^c = e^{c\ln z}$ computed that way **jumps** whenever $z$ crosses the negative real axis. Characteristic functions such as Heston's (Lesson 48) contain exactly such powers. A toy: $z(u) = e^{iu} + 0.3$ circles the origin, and $\sqrt{z(u)}$ should change smoothly with $u$. Keeping track of the angle as it winds (unwrapping it) removes the jumps.

In [ ]:
ub = np.linspace(0, 6 * math.pi, 1201)
zb = np.exp(1j * ub) + 0.3
naive = np.exp(0.5 * np.log(zb))                                 # principal logarithm
fixed = np.sqrt(np.abs(zb)) * np.exp(0.5j * np.unwrap(np.angle(zb)))
jumps = int((np.abs(np.diff(naive)) > 0.5).sum())
out["branch"] = {"jumps": jumps, "jumps_fixed": int((np.abs(np.diff(fixed)) > 0.5).sum()), "max_step_fixed": float(np.abs(np.diff(fixed)).max()),
                 "fig": {"u": ub[::4].round(4).tolist(), "naive": naive.imag[::4].round(4).tolist(), "fixed": fixed.imag[::4].round(4).tolist()}}
{k: v for k, v in out["branch"].items() if k != "fig"}

In [ ]:
# Chart data for the video (no random numbers)
out["complex"]["fig"]["ticks"] = [{"x": float(v), "label": f"{v}"} for v in (-4, -3, -2, -1, 0, 1)]
out["cf"]["fig"]["ticks"] = [{"x": float(v), "label": f"{v}"} for v in (0, 5, 10, 15)]
out["dens"]["fig"]["ticks"] = [{"x": round(math.log(v), 4), "label": f"{v}"} for v in (50, 70, 100, 140, 200)]
out["branch"]["fig"]["ticks"] = [{"x": round(k * math.pi, 4), "label": "0" if k == 0 else ("π" if k == 1 else f"{k}π")} for k in range(7)]

## 7. Exercises

Try each one before looking anything up. Full solutions are in the companion volume, *Part 5: Solutions*.

**Exercise 1.** Let $z = 3 + 4i$. Compute $z(1 - 2i)$, $|z|$ and $1/z$. Show that multiplying any $z$ by $i$ turns it a quarter turn anticlockwise, without changing its length.

**Exercise 2.** Derive Euler's formula $e^{i\theta} = \cos\theta + i\sin\theta$ from the Taylor series of $e^x$, $\cos x$ and $\sin x$. Deduce that $|e^{i\theta}| = 1$, and use $e^{i(a + b)} = e^{ia}e^{ib}$ to get the addition formulas for $\cos(a + b)$ and $\sin(a + b)$.

**Exercise 3.** Show that the characteristic function of $N(m, s^2)$ is $e^{ium - \frac12s^2u^2}$, by the completing-the-square calculation of Lesson 4 with $iu$ in place of a real number. Write down the characteristic function of $\ln S_T$ under the risk-neutral probability, with the course's numbers.

**Exercise 4.** Show that every characteristic function has $\varphi(0) = 1$, $|\varphi(u)| \le 1$ and $\varphi(-u) = \overline{\varphi(u)}$. Explain why $S_T$ has no moment generating function ($E[e^{tS_T}]$ is infinite for every $t > 0$), although $\ln S_T$ has a characteristic function.

**Exercise 5.** Lesson 27's share measure weights each outcome by $e^{X}/E[e^{X}]$, with $X = \ln S_T$. Show that the characteristic function of $X$ under it is $\varphi(u - i)/\varphi(-i)$. For the Black-Scholes $\varphi$, show that this is a normal characteristic function, and find its mean.

**Exercise 6.** Show that the Gil-Pelaez integrand $\mathrm{Re}\big[e^{-iuk}\varphi(u)/(iu)\big]$ tends to $E[X] - k$ as $u \to 0$, so the integral has no problem at 0. Evaluate this limit for $X = \ln S_T$ and $k = \ln K$ with the course's numbers.

**Exercise 7.** (Code) Price the call for strikes 80 to 130 in steps of 5 by Gil-Pelaez, and compare with the Black-Scholes formula. Find an upper limit $U$ and a step $du$ that give four correct decimals at every strike.

**Exercise 8.** (Code) For $z(u) = e^{iu} + 0.3$ on $0 \le u \le 6\pi$, compute $z^{1/2}$ with the principal logarithm and with an unwrapped angle. Count the jumps in each, and check that the unwrapped version changes smoothly. What would go wrong in a price that integrates such a function?

## 8. Export

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")